# 02 — Theme A: Probability Distributions & Sampling

**Owner:** _(put your name here)_ · **Branch:** `feature/theme-a`

## What this notebook argues

The allocation committee wants district-level malaria burden estimates. Two
statistical questions stand between us and that number:

1. **What distribution do the counts actually follow?** If we quote a burden
   estimate built on the wrong distribution, the *uncertainty* around it is
   wrong too — and this whole proposal turns on uncertainty.
2. **How wide is the interval around a survey-based estimate?** The household
   survey is a *cluster* sample. Treating it as a simple random sample makes our
   intervals look tighter than they are, which would make us overconfident about
   exactly the under-sampled rural districts that matter most for equity.

The four tasks map to those questions:

| Task | Question | Where |
|---|---|---|
| A1 | Are the district case counts over-dispersed? | §A1 |
| A2 | Does a Poisson match that shape? | §A2 |
| A3 | Does a Negative Binomial do better, and by how much? | §A3 |
| A4 | How wide is the real interval on a rural prevalence estimate? | §A4 |

## How to read it

Every number in the prose below is **printed by the cell above it**, not typed in
by hand. If you re-run on different data the sentences change with it. Cells
marked **“Your turn”** are judgement calls the panel will ask *you* about — write
your own sentence there.

> **Data rule.** This notebook prints aggregates, model output and figures only.
> No cell displays household records. See `RULES.md` §1.

---
## Setup

`RANDOM_SEED` is imported from `src/io.py` and passed explicitly to every
bootstrap. Nothing in this notebook draws unseeded randomness — a re-run
reproduces the intervals exactly.

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))  # run from notebooks/, import from src/
from src import io, models, uncertainty as unc, viz

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
warnings.filterwarnings("ignore", category=FutureWarning)
viz.set_theme()

SEED = io.RANDOM_SEED
N_BOOT = 2000  # bootstrap replicates; 2000 is enough for a 95% percentile interval
print(f"random seed = {SEED}, bootstrap replicates = {N_BOOT}")

In [ ]:
district = io.load_district_cases()
mis = io.load_mis_sample()
region_malaria = io.load_region_malaria()

# Shape and schema only — never rows (RULES.md §1).
for name, df in [("district_cases", district), ("mis_sample", mis), ("region_malaria", region_malaria)]:
    print(f"{name:16s} {df.shape[0]:>6,} rows x {df.shape[1]:>2} cols")

### Preflight: does the delivered data match the data dictionary?

Run this **first, every time**. `src/io.py` holds a column contract transcribed
from `data_dictionary.md`; this compares the files we actually received against
it. A mismatch is not something to code around — it is a finding for the
datasheet, and §A4 below depends on one of them.

It also flags **constant columns**, which are collinear with the intercept and
will make any GLM fit fail with an opaque `LinAlgError` rather than a useful
message.

In [ ]:
for name, df in [("ghana_mis_sample.csv", mis),
                 ("ghana_district_cases.csv", district),
                 ("ghana_region_malaria.csv", region_malaria)]:
    print(io.schema_summary(io.validate_schema(df, name)))
    print()

In [ ]:
# The two structural facts that decide how §A3 is specified.
print("distinct net_coverage_pct values:", district["net_coverage_pct"].nunique(),
      "| distinct region_code values:", district["region_code"].nunique())
print("-> if these are equal, net_coverage_pct IS the region variable and the two"
      " cannot both enter the model.\n")

missing_mis = io.validate_schema(mis, "ghana_mis_sample.csv").attrs["missing"]
print("parasitaemia column present:", "parasitaemia" in mis.columns,
      "| district identifier present:", "district" in mis.columns)
print("-> both are assumed by the case brief; see the discrepancy table in §A4.")

---
# A1 — Look at the response

**The response variable.** `positive_cases` from `ghana_district_cases.csv`:
confirmed malaria positives per district, cumulated over the reporting window
(2014–17), from Ghana Health Service routine surveillance.

**Why we expect over-dispersion before we look.** Districts differ enormously in
population, and burden concentrates in ways population alone does not explain
(transmission intensity, rainfall, existing net coverage). A Poisson process
assumes one single rate generates every count. Here the rate itself varies
between districts — and a mixture of Poissons with varying rates has more
variance than any single Poisson. That is over-dispersion, and it is a claim we
can check in one line: **is the variance bigger than the mean?**

In [ ]:
y = district["positive_cases"]
moments = models.moment_summary(y)
pd.Series(moments).to_frame("positive_cases")

In [ ]:
ratio = moments["variance_mean_ratio"]
print(
    f"Across {moments['n']} districts, positive cases range from "
    f"{moments['min']:,.0f} to {moments['max']:,.0f}.\n"
    f"Mean = {moments['mean']:,.0f}; variance = {moments['variance']:,.0f}.\n"
    f"Variance / mean = {ratio:,.1f}. Poisson requires this to be 1.0.\n"
    f"-> The counts are over-dispersed by a factor of roughly {ratio:,.0f}."
)

**Reading the ratio.** Under a Poisson, mean and variance are the *same
parameter* — so variance/mean = 1 is not a rule of thumb, it is the model's
definition. A ratio far above 1 is not a borderline test result; it says the
Poisson's core assumption is violated by orders of magnitude.

One caveat to state on the slide: this ratio is computed on **raw counts across
districts of very different sizes**, so part of it is simply population
variation, not extra-Poisson randomness. §A3 controls for population with an
exposure offset and re-checks the dispersion that survives. Both numbers belong
in the report — the raw one shows the shape, the adjusted one shows what is left
for the model to explain.

In [ ]:
fig, ax, edges = viz.plot_count_distribution(
    y, bins=15, title="District malaria positives, northern Ghana (2014–17)",
    xlabel="confirmed positive cases (cumulative)",
)
viz.save_figure(fig, "a1_count_distribution.png")

> **Your turn.** Describe the shape in one sentence you could say out loud to the
> panel — where is the mass, how long is the right tail, and does any district
> look like an outlier you would want to check against the raw workbook?
>
> _Your answer:_

---
# A2 — Fit a Poisson, and watch it fail

**The estimate.** For a Poisson, the maximum-likelihood estimate of the rate λ is
just the sample mean. The derivation is two lines: the log-likelihood of
$n$ counts is $\ell(\lambda)=\sum_i (y_i\log\lambda - \lambda) + c$; setting
$\partial\ell/\partial\lambda = \sum_i y_i/\lambda - n = 0$ gives
$\hat\lambda = \bar y$. No optimiser needed.

**How we compare it to the data.** We integrate the fitted pmf over each
histogram bin (via the CDF) rather than evaluating it at bin centres. With bins
this wide relative to the pmf, evaluating at centres would understate the
model's mass and make the comparison unfair to the Poisson. We want the
comparison to be honest, then let it fail on its merits.

In [ ]:
lam_hat = float(y.mean())
expected_poisson = models.poisson_pmf_expected(y, edges, lam=lam_hat)

fig, ax, edges = viz.plot_count_distribution(
    y, bins=15, title=f"Observed counts vs fitted Poisson(λ = {lam_hat:,.0f})",
    xlabel="confirmed positive cases (cumulative)",
)
viz.overlay_expected(ax, edges, expected_poisson, f"Poisson(λ={lam_hat:,.0f})", "#C44E52")
viz.save_figure(fig, "a2_poisson_overlay.png")

In [ ]:
sd_poisson = np.sqrt(lam_hat)
sd_observed = float(y.std(ddof=1))
print(
    f"Poisson(λ={lam_hat:,.0f}) implies a standard deviation of sqrt(λ) = {sd_poisson:,.0f}.\n"
    f"The districts actually vary with a standard deviation of {sd_observed:,.0f}"
    f" — about {sd_observed / sd_poisson:,.0f}x wider.\n"
    f"Fitted Poisson mass inside the plotted range: {expected_poisson.sum():.1f} of {len(y)} districts."
)

**The verdict.** The fitted Poisson is a near-vertical spike: it concentrates
essentially all of its probability in a narrow band around the mean, because its
spread is locked to `sqrt(λ)` and cannot be tuned. The observed counts spread out
roughly two orders of magnitude further. The model does not just fit poorly, it
**has no free parameter capable of fitting this**.

This is why the Poisson is in the report at all: not as a candidate, but as the
demonstration of *why we need the extra parameter* that the Negative Binomial
adds. (CLAUDE.md, statistical conventions.)

---
# A3 — Negative Binomial

The NB2 parameterisation adds one dispersion parameter α and lets the variance
grow quadratically with the mean:

$$\mathrm{Var}(Y) = \mu + \alpha\mu^2$$

Poisson is the **α → 0 special case**, which makes the comparison a clean nested
one: α measures exactly how much extra-Poisson variation the data demand.

### A3.1 — α from the moments, before any optimiser

Rearranging the variance function on the sample moments gives
$\hat\alpha = (s^2 - \bar y)/\bar y^2$. This is the number to put on the slide if
someone asks where α comes from — it is algebra, not a black box.

In [ ]:
alpha_mom = models.nb_alpha_from_moments(y)
expected_nb = models.nbinom_pmf_expected(y, edges, alpha=alpha_mom)

print(f"Method-of-moments alpha (marginal, no covariates) = {alpha_mom:.4f}")
print(f"Implied NB sd at the mean = {np.sqrt(lam_hat + alpha_mom * lam_hat**2):,.0f}"
      f"  (observed {sd_observed:,.0f}, Poisson {sd_poisson:,.0f})")
print(f"NB mass inside the plotted range: {expected_nb.sum():.1f} of {len(y)} districts"
      f" — the remainder is right-tail mass beyond the axis, which is the point.")

In [ ]:
fig, ax, edges = viz.plot_count_distribution(
    y, bins=15, title="Observed counts: Poisson vs Negative Binomial",
    xlabel="confirmed positive cases (cumulative)",
)
viz.overlay_expected(ax, edges, expected_poisson, "Poisson", "#C44E52")
viz.overlay_expected(ax, edges, expected_nb, f"Negative Binomial (α={alpha_mom:.3f})", "#55A868")
viz.save_figure(fig, "a3_poisson_vs_nb.png")

### A3.2 — The same comparison as a regression, with population as exposure

Comparing raw counts across districts of wildly different populations conflates
two things: *how much malaria there is* and *how many people there are*. The fix
is an **exposure offset**: we model

$$\log \mathbb{E}[y_i] = \log(\text{population}_i) + \beta_0 + \beta_1 x_i$$

which is equivalent to modelling the **rate** while keeping the response on the
count scale where the Poisson/NB likelihoods live. The offset coefficient is
fixed at 1 — it is not estimated, so it costs no degrees of freedom.

> **Justification for the record:** we use `mean_population` as exposure because
> `positive_per_100k` is already a ratio, and modelling a ratio as if it were a
> count would misstate the likelihood. Same information, correct distribution.

**Before fitting: check the design matrix.** Two columns in this file will
silently break the fit, and the data dictionary predicts both:

- `net_coverage_pct` is a **region-level** figure joined onto districts, so it
  takes one value per region and is perfectly collinear with a region dummy.
- `months_reported` may be constant across districts, in which case it is
  collinear with the intercept.

Either one makes the Hessian singular, and statsmodels reports that as an opaque
`LinAlgError` rather than "your model is unidentified". `check_design_matrix`
catches it first and tells us which column is at fault.

In [ ]:
candidate = "positive_cases ~ net_coverage_pct + months_reported + C(region_code)"
report = models.check_design_matrix(candidate, district)
print(f"design columns = {report.attrs['n_columns']}, matrix rank = {report.attrs['rank']}, "
      f"rank deficient = {report.attrs['rank_deficient']}")
report

In [ ]:
# Keep region OR net_coverage_pct, not both, and drop any constant column.
formula = "positive_cases ~ net_coverage_pct"
final_report = models.check_design_matrix(formula, district)
print(f"'{formula}' -> rank deficient: {final_report.attrs['rank_deficient']}")

offset = np.log(district["mean_population"].clip(lower=1)).to_numpy()
poisson_fit = models.fit_poisson(formula, district, offset=offset)
nb_fit = models.fit_negative_binomial_mle(formula, district, offset=offset)
print(f"NB converged: {nb_fit.mle_retvals.get('converged')}")

> **Finding to carry into the leakage audit (B4).** Because `net_coverage_pct`
> resolves to one value per region, its coefficient here is a **region effect
> wearing a net-coverage label**. We must not report it as "districts with more
> nets have fewer cases" — with three distinct values there is no district-level
> variation to support that claim. It is also measured in **2022**, while the
> case counts are **2014–17**, so the arrow of time points the wrong way.

In [ ]:
dispersion = models.check_dispersion(poisson_fit)
print(f"Poisson GLM Pearson χ²/df = {dispersion['dispersion_ratio']:,.1f} "
      f"(χ² = {dispersion['pearson_chi2']:,.0f} on {dispersion['df_resid']:.0f} df)")
print(f"NB alpha (MLE, offset model) = {float(nb_fit.params['alpha']):.4f}")
models.compare_fits(poisson_fit, nb_fit).round(1)

In [ ]:
d_aic = models.compare_fits(poisson_fit, nb_fit)["AIC"]
print(
    f"Even after population is accounted for by the offset, the Poisson's dispersion\n"
    f"ratio is {dispersion['dispersion_ratio']:,.0f} — a correctly specified Poisson gives ~1.\n\n"
    f"AIC: Poisson {d_aic['Poisson']:,.0f} vs Negative Binomial {d_aic['Negative Binomial']:,.0f}\n"
    f"(difference of {d_aic['Poisson'] - d_aic['Negative Binomial']:,.0f} for one extra parameter)."
)

**Why the two α values differ.** The marginal α in §A3.1 and the MLE α here are
answering different questions. The marginal one measures *all* the spread between
districts, including the fact that some districts are ten times larger than
others. The offset model has already explained that part, so its α measures only
the dispersion **left over** after population is accounted for. The smaller
number is the honest one to quote for the model; the larger one describes the raw
data. Say which is which when you present them.

**Conclusion for Theme A.** The Negative Binomial is the working model. We report
it with α and an interval; the Poisson stays in the appendix as the demonstration
of why one parameter was not enough.

---
# A4 — How wide is the interval really?

### A discrepancy to report first

The case brief describes the survey extract as containing *"cluster, rural, net
usage, parasitaemia, sample weight"*. The delivered file does **not** contain a
parasitaemia column, and it has **no district identifier** — the finest
geography is `region` (`hv024`) plus the anonymised `cluster` id.

We are not working around this silently (CLAUDE.md). We adapt and say so:

| Brief asks for | File provides | What we do |
|---|---|---|
| parasitaemia prevalence | `has_net`, `num_nets`, `children_under_net_last_night` | Estimate **net ownership** — a proportion with the same cluster structure, so the sampling lesson is identical. Prevalence itself comes from `ghana_region_malaria.csv`, which ships published CIs. |
| an under-sampled **district** | region + cluster only | Use the **rural households of the region with the fewest sampled clusters** as the under-sampled domain, and say "region", not "district", in the report. |

This belongs in `reports/datasheet.md` as a data-availability caveat.

In [ ]:
# Region labels: the 2022 rows of the DHS file carry the 16-region coding.
region_names = (
    region_malaria.query("survey_year == 2022")
    .dropna(subset=["hv024_16region"])
    .set_index("hv024_16region")["region_name"]
    .to_dict()
)

rural = mis[mis["residence"] == "rural"]
coverage = (
    rural.groupby("region")
    .agg(n_clusters=("cluster", "nunique"), n_households=("household", "size"))
    .sort_values("n_clusters")
)
coverage.insert(0, "region_name", [region_names.get(r, f"region {r}") for r in coverage.index])
coverage.head(5)

> **Check the domain before you use it.** "Fewest rural clusters" can select a
> region that is simply *mostly urban* rather than one that is rurally
> under-served — a very different thing, and not the equity story we want to
> tell. Look at the table above: if the top row is a metropolitan region, pick the
> lowest-ranked region that is substantively rural and say in the report why you
> overrode the automatic choice. Set `target_region` by hand in that case.

In [ ]:
target_region = int(coverage.index[0])
target_name = coverage.iloc[0]["region_name"]
domain = rural[rural["region"] == target_region]

print(f"Under-sampled domain: rural {target_name} (hv024 = {target_region})")
print(unc.summarise_sample_size(domain))

### Why the cluster bootstrap, and what the naive one gets wrong

DHS is a **two-stage** design: clusters (enumeration areas) are drawn first, then
households within them. Households in the same cluster are not independent
observations — they share a village, a water body, a vector environment and
whichever distribution campaign last reached them. Their outcomes are positively
correlated.

A household-level bootstrap resamples as if all *n* households were independent
draws. That assumes more independent information than the survey actually bought,
so it produces an interval that is **too narrow**. The cluster bootstrap
resamples the *clusters* first, which is what lets the between-cluster variation
into the interval.

We run both — not because both are defensible, but because the gap between them
**is the finding**.

In [ ]:
def weighted_net_ownership(df):
    "Survey-weighted % of households owning at least one net."
    return unc.weighted_proportion(df, value_col="has_net", weight_col="sample_weight")

ci_table = unc.compare_bootstraps(
    domain, weighted_net_ownership, label=f"rural {target_name}",
    n_bootstraps=N_BOOT, random_seed=SEED,
)
deff = ci_table.attrs["design_effect"]
ci_table.round(2)

In [ ]:
naive, cluster = ci_table.iloc[0], ci_table.iloc[1]
print(
    f"Rural {target_name}: weighted net ownership = {cluster['estimate']:.1f}%\n\n"
    f"  Naive household bootstrap : [{naive['ci_low']:.1f}, {naive['ci_high']:.1f}]  "
    f"width {naive['ci_width']:.1f} pp   <-- WRONG for this design\n"
    f"  Cluster bootstrap         : [{cluster['ci_low']:.1f}, {cluster['ci_high']:.1f}]  "
    f"width {cluster['ci_width']:.1f} pp   <-- what we report\n\n"
    f"The correct interval is {cluster['ci_width'] / naive['ci_width']:.2f}x wider.\n"
    f"Implied design effect (ratio of variances) = {deff:.2f}: this cluster sample carries\n"
    f"the information of a simple random sample about {1/deff:.0%} its size."
)

In [ ]:
fig, ax = viz.plot_ci_comparison(
    ci_table, title=f"95% CI for net ownership — rural {target_name} ({N_BOOT} replicates)"
)
viz.save_figure(fig, "a4_bootstrap_ci_comparison.png")

### A4.1 — Checking our method against the published reference

The data dictionary promises something valuable: *"The `_ci_low`/`_ci_high`
columns are the official design-based intervals, so you can compare your own
bootstrap intervals (Task A4) against them — a rare chance to check your method
against the reference."*

Check that the columns are actually populated before relying on them.

In [ ]:
ci_cols = [c for c in region_malaria.columns if c.endswith(("_ci_low", "_ci_high"))]
coverage_ci = region_malaria[ci_cols].notna().sum().to_frame("non_null_rows")
coverage_ci["of_total"] = len(region_malaria)
coverage_ci

> **Discrepancy #2 — the ITN confidence intervals are empty.**
> `net_ownership_pct_ci_low/high` and `u5_itn_use_pct_ci_low/high` are present as
> columns but contain **no values at all**. The prevalence CIs
> (`rdt_prevalence_pct_ci_*`) *are* populated, from 2014 onward.
>
> So the promised check does not close, and the reason is structural:
>
> | | published CI available? | can we bootstrap it? |
> |---|---|---|
> | Net ownership | **no** — column empty | yes (`has_net` is in the survey file) |
> | RDT prevalence | yes (2014+) | **no** — no parasitaemia column (Discrepancy #1) |
>
> The two halves never meet. This is the same gap as §A4's first discrepancy
> showing up a second time, and both belong in `reports/datasheet.md`.

**What we can still validate — and it is not nothing.** DHS publishes the
net-ownership *point estimate*. If our survey-weighted, cluster-bootstrapped
estimate reproduces it, that validates our **weighting and estimation**, even
though it cannot validate our **interval**. Say exactly that much on the slide,
and no more.

In [ ]:
region_all = mis[mis["region"] == target_region]
est, lo, hi = unc.cluster_bootstrap_ci(
    region_all, weighted_net_ownership, n_bootstraps=N_BOOT, random_seed=SEED
)

published = region_malaria.query("survey_year == 2022 and hv024_16region == @target_region")
pub_pt = float(published["net_ownership_pct"].iloc[0]) if len(published) else float("nan")
pub_lo = float(published["net_ownership_pct_ci_low"].iloc[0]) if len(published) else float("nan")

print(f"Our cluster bootstrap, all {target_name}: {est:.1f}%  [{lo:.1f}, {hi:.1f}]  "
      f"(width {hi - lo:.1f} pp)")
print(f"DHS published point estimate:          {pub_pt:.1f}%", end="  ")
print("[no published CI in this file]" if np.isnan(pub_lo) else f"[{pub_lo:.1f}, ...]")
print(f"\nDifference in point estimate: {abs(est - pub_pt):.2f} pp")

> **Your turn.** How close is our point estimate to the published one, and what
> does that let you claim? Write the claim at the strength the evidence supports
> — "our weighted estimator reproduces the DHS published figure to within X pp"
> is defensible; "our confidence interval is validated" is not, because there is
> no published interval to compare against.
>
> _Your answer:_

---
# What Theme A hands to the rest of the project

**To the model (Theme B / notebook 03):** the response is over-dispersed, so any
predictive model of district burden is a **Negative Binomial**, fitted with
`log(population)` as offset. A Poisson would produce standard errors that are far
too small and district rankings that look more certain than they are.

**To the allocation (notebook 04):** the correct interval on a rural survey
estimate is materially wider than a naive one. Districts in under-sampled regions
therefore have *genuinely* less certain burden estimates. The equity argument
follows directly: a district can be a priority because its burden is high, **or**
because our interval on it is so wide that we cannot rule out a high burden. The
mean-best ranking hides the second group.

**To the leakage audit (B4):** `net_coverage_pct` is region-level and measured in
2022, while the case counts are 2014–17. Using it to explain those counts is both
a resolution mismatch and a temporal one.

**To the datasheet (LO7):** the delivered survey extract has no parasitaemia
column and no district identifier, contrary to the brief.

In [ ]:
claims = pd.DataFrame([
    {"id": "C-02", "claim": f"District positive counts are over-dispersed (variance/mean = {ratio:,.0f}, "
                            f"Poisson requires 1.0)", "source": "ghana_district_cases.csv", "cell": "A1"},
    {"id": "C-02b", "claim": f"Over-dispersion survives population adjustment (Poisson GLM Pearson χ²/df = "
                             f"{dispersion['dispersion_ratio']:,.0f} with log-population offset)",
     "source": "ghana_district_cases.csv", "cell": "A3.2"},
    {"id": "C-02c", "claim": f"NB beats Poisson by {d_aic['Poisson'] - d_aic['Negative Binomial']:,.0f} AIC "
                             f"for one extra parameter (α = {float(nb_fit.params['alpha']):.4f})",
     "source": "ghana_district_cases.csv", "cell": "A3.2"},
    {"id": "C-03", "claim": f"Cluster bootstrap CI is {cluster['ci_width'] / naive['ci_width']:.2f}x wider than "
                            f"naive (DEFF ≈ {deff:.2f}) for rural {target_name} net ownership",
     "source": "ghana_mis_sample.csv", "cell": "A4"},
])
claims.to_markdown(index=False)  # paste into reports/claims_table.md

---
### Before you commit

```bash
jupyter nbconvert --clear-output --inplace notebooks/02_distributions.ipynb
black src/ && ruff check src/
```

Then add a `WORKLOG.md` entry at the top: branch, assistant used and for what,
what changed, decisions, blockers.